Test Set Inference with Best Model

Loads the best model saved by `Model_V4.ipynb` and runs it on the held out test set.
Saves a CSV with columns: `sequence`, `aa_length`, `label`, `prob_amp`, `predicted`, `correct`, `result`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Thesis_V4

!pip install -q -r requirements.txt

Mounted at /content/drive
/content/drive/MyDrive/Thesis_V4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 65.3 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [ ]:
BASE_DIR        = '/content/drive/MyDrive/Thesis_V4'
DATA_PATH       = f'{BASE_DIR}/all_veltri.csv'
MODEL_PATH      = f'{BASE_DIR}/best_model'
OUTPUT_CSV      = f'{BASE_DIR}/test_predictions.csv'

THRESHOLD = 0.480

print(f'Model path : {MODEL_PATH}')
print(f'Output CSV : {OUTPUT_CSV}')
print(f'Threshold  : {THRESHOLD}')

Model path : /content/drive/MyDrive/Thesis_V4/best_model
Output CSV : /content/drive/MyDrive/Thesis_V4/test_predictions.csv
Threshold  : 0.48


In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0)
df['AMP']     = df['AMP'].astype(bool).astype(int)
df            = df.sample(frac=1, random_state=42).reset_index(drop=True)
df['seq_len'] = df['aa_seq'].str.len()

train_df, temp_df = train_test_split(
    df, test_size=0.4, stratify=df['AMP'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['AMP'], random_state=42
)

print(f'Total rows : {len(df)}')
print(f'Train      : {len(train_df)}')
print(f'Validation : {len(val_df)}')
print(f'Test       : {len(test_df)}')
print('Test label counts:')
print(test_df['AMP'].value_counts().to_string())

Total rows : 3556
Train      : 2133
Validation : 711
Test       : 712
Test label counts:
AMP
0    356
1    356


In [ ]:
print('Loading tokeniser…')
tokenizer = BertTokenizer.from_pretrained(
    MODEL_PATH, do_lower_case=False, use_fast=False
)
print('Tokeniser ready.')


class AMPData(Dataset):
    """Identical to Model_V4 — space-separated AAs for ProtBERT."""

    def __init__(self, df, tokenizer, max_len=200):
        self.seqs    = list(df['aa_seq'])
        self.labels  = list(df['AMP'].astype(int))
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = ' '.join(self.seqs[idx])
        enc = self.tok(
            seq,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item


test_dataset = AMPData(test_df, tokenizer)
print(f'Test dataset size: {len(test_dataset)}')

Loading tokeniser…
Tokeniser ready.
Test dataset size: 712


In [ ]:
print(f'Loading model from {MODEL_PATH} …')
model = BertForSequenceClassification.from_pretrained(
    MODEL_PATH, ignore_mismatched_sizes=True
)
model.eval()
print('Model loaded.')

Loading model from /content/drive/MyDrive/Thesis_V4/best_model …


Loading weights:   0%|          | 0/489 [00:02<?, ?it/s]

Model loaded.


In [ ]:
trainer = Trainer(model=model)

test_out    = trainer.predict(test_dataset)
test_labels = test_out.label_ids
test_probs  = torch.softmax(
    torch.tensor(test_out.predictions, dtype=torch.float32), dim=-1
)[:, 1].numpy()
test_preds  = (test_probs >= THRESHOLD).astype(int)

print(f'Predictions complete. Samples: {len(test_labels)}')
print(f'Positive predictions: {test_preds.sum()} / {len(test_preds)}')

Predictions complete. Samples: 712
Positive predictions: 358 / 712


In [ ]:
sequences = test_df['aa_seq'].reset_index(drop=True).tolist()

def classify_result(true_label, pred_label):
    if true_label == 1 and pred_label == 1:
        return 'TP'
    elif true_label == 0 and pred_label == 1:
        return 'FP'
    elif true_label == 0 and pred_label == 0:
        return 'TN'
    else:
        return 'FN'

results_df = pd.DataFrame({
    'sequence' : sequences,
    'aa_length': [len(s) for s in sequences],
    'label'    : test_labels.tolist(),
    'prob_amp' : np.round(test_probs, 6).tolist(),
    'predicted': test_preds.tolist(),
    'correct'  : (test_labels == test_preds).astype(int).tolist(),
    'result'   : [
        classify_result(int(t), int(p))
        for t, p in zip(test_labels, test_preds)
    ],
})

print(results_df.head(10).to_string())
print(f'\nResult counts:')
print(results_df['result'].value_counts().to_string())

                                                                                   sequence  aa_length  label  prob_amp  predicted  correct result
0                                                                    VVRVGGRVPVLAGTGLSGTAKT         22      0  0.276276          0        1     TN
1                                                                       GIVDFAKKVVGGIRNALGI         19      1  0.799875          1        1     TP
2                                                                             FISAIASMLGKFL         13      1  0.799397          1        1     TP
3                                                                             FLPLVGKILSGLI         13      1  0.798918          1        1     TP
4                                                                       GLLSALRKMIPHILSHIKK         19      1  0.800874          1        1     TP
5                                                                      AALKGCWTKSIPPKPCSGKR         20      1  0.79957

In [ ]:
counts = results_df['result'].value_counts()
TP = counts.get('TP', 0)
FP = counts.get('FP', 0)
TN = counts.get('TN', 0)
FN = counts.get('FN', 0)

accuracy  = (TP + TN) / (TP + FP + TN + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else float('nan')
recall    = TP / (TP + FN) if (TP + FN) > 0 else float('nan')
f1        = (2 * precision * recall / (precision + recall)
             if (precision + recall) > 0 else float('nan'))

print('=' * 45)
print('  TEST SET RESULTS')
print('=' * 45)
print(f'  Threshold : {THRESHOLD}')
print(f'  TP        : {TP}')
print(f'  FP        : {FP}')
print(f'  TN        : {TN}')
print(f'  FN        : {FN}')
print(f'  Accuracy  : {accuracy:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print('=' * 45)

  TEST SET RESULTS
  Threshold : 0.48
  TP        : 323
  FP        : 35
  TN        : 321
  FN        : 33
  Accuracy  : 0.9045
  Precision : 0.9022
  Recall    : 0.9073
  F1 Score  : 0.9048


In [ ]:
results_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(results_df)} rows → {OUTPUT_CSV}')
print('\nColumn dtypes:')
print(results_df.dtypes.to_string())

Saved 712 rows → /content/drive/MyDrive/Thesis_V4/test_predictions.csv

Column dtypes:
sequence      object
aa_length      int64
label          int64
prob_amp     float64
predicted      int64
correct        int64
result        object
